In [1]:
import pandas as pd
import numpy as np
from utils import concat_data_across_years, merge_with_or

import warnings
warnings.filterwarnings('ignore')

# User Data Preparation  

This notebook aggregates raw data from [NHANES dataset](https://wwwn.cdc.gov/nchs/nhanes/). Place the data folder as required and running the notebook will generate a list of processed tables of aggregated information. Specially:

 * `main_table.csv`: This is main table where the full SEQN list is stored. It also contains basic demograph information of the users.

 * `user_tagging.csv`: This table is indexed by SEQN and shows the nutrition tags given to the users.

 * `user_info_data.csv`: This big table contains all relevant info on the user side, which is the main source of the next phase of the pipeline.    
 
 * Other tables including `user_habit.csv`, `user_medical.csv`, etc. are also indexed by SEQN and show the specific aspects of the users' informaiton.

In [2]:
# NHANES data is updated every two years, and '0304' means the data is from 2003-2004.
years = ['0304', '0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720']
# This char is used by NHANES for the dataset name of each year.
year_char = 'C'

### 1. Collect Demographic Information

In [3]:
type_demo = 'demographic'
df_demo = concat_data_across_years(type_demo, 'DEMO', years, year_char)
# 95872 unique records in total

In [4]:
# Select the wanted columns. Make changes here if needed in the future.
df_demo = df_demo[['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'INDHHINC', 'DMDEDUC2', 'WTINT2YR',
                       'WTMEC2YR', 'WTINTPRP', 'WTMECPRP', 'years']]
df_demo = df_demo.fillna(0)
df_demo[['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'INDHHINC', 'DMDEDUC2']] = \
    df_demo[['SEQN', 'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'INDHHINC', 'DMDEDUC2']].astype(int)

# Unify the weight columns and make column names readable.
df_demo['weight_interview'] = np.where(df_demo['WTINT2YR'] == -1, df_demo['WTINTPRP'], df_demo['WTINT2YR'])
df_demo['weight_mec'] = np.where(df_demo['WTMEC2YR'] == -1, df_demo['WTMECPRP'], df_demo['WTMEC2YR'])
df_demo.drop(['WTINT2YR','WTMEC2YR', 'WTINTPRP', 'WTMECPRP'], axis=1, inplace=True)
df_demo = df_demo.rename(columns={'RIAGENDR': 'gender', 'RIDAGEYR': 'age', 'RIDRETH1': 'race', 'DMDEDUC2': 'education',
                                  'INDHHINC': 'household_income'})
df_demo['SEQN'] = df_demo['SEQN'].astype(str)
df_demo = df_demo.set_index('SEQN')

# Transform ages to age groups
bins = [-1, 10, 20, 30, 40, 50, 60, 100]
labels = ['1', '2', '3', '4', '5', '6', '7']

# Create a new column for age groups
df_demo['age_group'] = pd.cut(df_demo['age'], bins=bins, labels=labels, right=True)

### 2. Questionnaire Data

In [5]:
type_questionnaire = 'questionnaire'

#### 2.1. Diet Behaviors & Nutrition

In [6]:
'''
TODO: We need to integrate this habit informaiton into our pipeline. 
The remaining issues are: 
1) Integrate the pipeline so user_habit table can be generated here.
2) Make sure the raw data required is updated. Some data is not properly downloaded to data folder.
3) Check the percentiles and make sure it's the right one.
4) There are too many print statements. Clean them up.

Check the habit generation for dietary habit information.
View: https://drive.google.com/drive/folders/1FAQGkGkjWMiaDWHZ_zBRivzGSdg85NT-?usp=sharing
'''

"\nTODO: We need to integrate this habit informaiton into our pipeline. \nThe remaining issues are: \n1) Integrate the pipeline so user_habit table can be generated here.\n2) Make sure the raw data required is updated. Some data is not properly downloaded to data folder.\n3) Check the percentiles and make sure it's the right one.\n4) There are too many print statements. Clean them up.\n\nCheck the habit generation for dietary habit information.\nView: https://drive.google.com/drive/folders/1FAQGkGkjWMiaDWHZ_zBRivzGSdg85NT-?usp=sharing\n"

#### 2.2. Drug Use

In [7]:
# No specific illicit drug info in 03-04 data. No such table in 17-20 data.
df_DU = concat_data_across_years(type_questionnaire, 'DUQ',
                                 ['0506', '0708', '0910', '1112', '1314', '1516', '1718'], 'D')

df_DU = df_DU.loc[df_DU['DUQ290'] == 1]
df_DU = df_DU[['SEQN', 'DUQ270U', 'DUQ350U', 'DUQ300', 'DUQ310Q', 'DUQ310U']]
df_DU = df_DU.fillna(-1)
df_DU = df_DU.astype(int)

"""
# We only care about those who have used heroin at least once.
# If within a year, the user has been using any illicit drugs (heroin, meth, cocaine),
# we identify the user as an active user.
# Otherwise, we consider this user a recovered user.

# To be specific, -1 means missing, 4 means it has been years that a user haven't used a drug.
# If there is a value that is neither -1, nor 4 in any of the three columns, the user is an active user.

# We label the active user as 1 and recovered user as 2.
# And the rest of the users who hasn't even used heroin or other opioid prescription drugs as 0: non-opioid-user
"""
df_DU['active_user'] = np.where((df_DU['DUQ270U'].isin([4, -1]) == False) |
                                 (df_DU['DUQ350U'].isin([4, -1]) == False) |
                                 (df_DU['DUQ310U'].isin([4, -1]) == False), 1, 2)

df_DU = df_DU.rename(columns={'DUQ300': 'age_first_use_heroin', 'DUQ310U': 'last_time_unit_used_heroin',
                              'DUQ310Q': 'last_time_used_heroin', 'DUQ270U': 'last_time_unit_used_cocaine',
                              'DUQ350U': 'last_time_unit_used_meth'})

"""
last_time_unit_used_heroin and last_time_used_heroin is a combo feature.
time unit defines whether it's years, months or days we are talking about.
the time defines the exact number of that unit.
For example, if time unit is 4 and time is 30, it means this user last used heroin 30 years ago.
"""
df_DU = df_DU.set_index('SEQN')

# This table records the use of illicit drugs. If we want to check people's age when they first used illicit drugs, we can do:
# df_drug_age = df_demo.merge(df_DU, how='right', left_on=df_demo.index, right_on=df_DU.index)

#### 2.3. Prescription Medicine

In [8]:
df_PM = concat_data_across_years(type_questionnaire, 'RXQ_RX', years, year_char)
df_PM = df_PM[df_PM['RXDUSE']==1]
df_PM['SEQN'] = df_PM['SEQN'].astype(int).astype(str)

# This creates a table that records the prescription medicines and their durations that each user has taken.
# df_PM[['SEQN', 'RXDDRUG', 'RXDDRGID', 'RXDDAYS']].to_csv('../processed_data/user_prescription_medicine.csv', index=False)

df_PM_1 = df_PM.loc[(df_PM['RXDRSC1'] == 'F11.2') | (df_PM['RXDRSC1'] == 'F11.23')]

# This is the prescription medicines that categorized as opioid. 
drugs = pd.read_sas('../data/RXQ_DRUG.xpt', encoding='ISO-8859-1')
drug_60 = drugs[(drugs['RXDDCI1A'] == 57) & (drugs['RXDDCI1B'] == 58) & (drugs['RXDDCI1C'] == 60)]
drug_191 = drugs[(drugs['RXDDCI1A'] == 57) & (drugs['RXDDCI1B'] == 58) & (drugs['RXDDCI1C'] == 191)]
drug = pd.concat([drug_60, drug_191])
drug_id = set(drug['RXDDRGID'].tolist())

df_PM  = df_PM [df_PM ['RXDDRGID'].isin(drug_id)]
df_PM = pd.concat([df_PM, df_PM_1])
df_PM = df_PM.drop_duplicates()
# 3992 records for taking opioid prescription or taking prescription for opioid dependence.

"""
# We define the long term opioid users as those how have taken opioid prescriptions over 90 days.

# Note that each user can take multiple opioid prescriptions.

# In an earlier study for tracking long term opioid users,
# The author excluded medications containing buprenorphine since they are used to treat use disorder.
# However, we find multiple cases that this medicine used to treat opioid dependence.
# So technically this also implies the user is a long term opioid user.
"""
df_PM = df_PM[['SEQN', 'RXDDRUG', 'RXDDRGID', 'RXDDAYS']]
df_PM = df_PM[df_PM['RXDDAYS'] > 90]
df_PM[['SEQN', 'RXDDAYS']] = df_PM[['SEQN', 'RXDDAYS']].astype(int)
df_PM = df_PM.rename(columns={'RXDDRUG': 'drug_name', 'RXDDRGID': 'drug_id', 'RXDDAYS': 'days_using'})
df_PM = df_PM.set_index('SEQN')

#### Create opioid labels for all users in the main table

In [9]:
opioid_user_set = set(df_PM.index.tolist())
"""
Following the labeling scheme, we get the label in the main table.
0: user, 1: active user, 2: recovered_user
"""

# For historical reasons, this label is stored in the main table as `labels`. For current work, we change it to `opioid_label`.
# This serves no purpose here, but leave a back door for our next work. 
df_demo['opioid_label'] = 0
df_demo['opioid_label'] = df_demo.index.map(df_DU['active_user']).fillna(0)
df_demo.loc[df_demo.index.isin(opioid_user_set), 'opioid_label'] = 1
# we have 92723 regular users, 2728 active opioid users, 219 of which are active heroin users, and 421 recovered users.

df_demo.to_csv('../processed_data/main_table.csv')

### 3. Laboratory Data

In [10]:

type_laboratory = 'laboratory'

#### 3.1. Standard Biochemistry Profile

In [11]:
df_SBP = concat_data_across_years(type_laboratory, 'BIOPRO',
                                 ['0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720'], 'D')
df_temp = concat_data_across_years(type_laboratory, 'L40',
                                 ['0304'], 'C')
df_SBP = pd.concat([df_SBP, df_temp])

# We keep all the items in this laboratory but filter out the same value with different unit.
# For example, we keep "Total calcium (mmol/L)" and get rid of "Total calcium (mg/dL)"
columns_SBP = [
    'SEQN',
    'LBDSALSI', 'LBXSATSI', 'LBXSASSI', 'LBXSAPSI', 'LBDSBUSI', 'LBDSCASI',
    'LBDSCHSI', 'LBXSC3SI', 'LBDSCRSI', 'LBXSGTSI', 'LBDSGLSI', 'LBDSIRSI',
    'LBXSLDSI', 'LBDSPHSI', 'LBDSTBSI', 'LBDSTPSI', 'LBDSTRSI', 'LBDSUASI',
    'LBDSCRSI', 'LBXSNASI', 'LBXSKSI', 'LBXSCLSI', 'LBXSOSSI', 'LBDSGBSI'
]
rename_SBP = {
    'LBDSALSI': 'Albumin (g/L)', 'LBXSATSI': 'Alanine aminotransferase (ALT) (U/L)',
    'LBXSASSI': 'Aspartate aminotransferase (AST) (U/L)', 'LBXSAPSI': 'Alkaline phosphatase (U/L)',
    'LBDSBUSI': 'Blood urea nitrogen (mmol/L)', 'LBDSCASI': 'Total calcium (mmol/L)',
    'LBDSCHSI': 'Cholesterol (mmol/L)', 'LBXSC3SI': 'Bicarbonate (mmol/L)',
    'LBDSCRSI': 'Creatinine (µmol/L)', 'LBXSGTSI': 'Gamma glutamyl transferase (U/L)',
    'LBDSGLSI': 'Glucose, serum (mmol/L)', 'LBDSIRSI': 'Iron, refrigerated (umol/L)',
    'LBXSLDSI': 'Lactate dehydrogenase LDH (U/L)', 'LBDSPHSI': 'Phosphorus (mmol/L)',
    'LBDSTBSI': 'Bilirubin, total (umol/L)', 'LBDSTPSI': 'Total protein (g/L)',
    'LBDSTRSI': 'Triglycerides (mmol/L)', 'LBDSUASI': 'Uric acid (umol/L)',
    'LBXSNASI': 'Sodium (mmol/L)',
    'LBXSKSI': 'Potassium (mmol/L)', 'LBXSCLSI': 'Chloride (mmol/L)',
    'LBXSOSSI': 'Osmolality (mmol/Kg)', 'LBDSGBSI': 'Globulin (g/L)'
}

# Not every respondent take this examination. In the 95872 respondents from 2003-2020, we have 60331 valid records.
df_SBP = df_SBP[columns_SBP]
df_SBP = df_SBP.rename(columns=rename_SBP)
df_SBP['SEQN'] = df_SBP['SEQN'].astype(int)
df_SBP = df_SBP.set_index('SEQN')
df_SBP.dropna(how='all', inplace=True)

#### 3.2.Complete Blood Count

In [12]:

df_CBC = concat_data_across_years(type_laboratory, 'CBC',
                                 ['0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720'], 'D')
df_temp = concat_data_across_years(type_laboratory, 'L25',
                                 ['0304'], 'C')
df_CBC = pd.concat([df_CBC, df_temp])

columns_CBC = [
    'SEQN',
    'LBXWBCSI', 'LBXLYPCT', 'LBXMOPCT', 'LBXNEPCT', 'LBXEOPCT',
    'LBXBAPCT', 'LBDLYMNO', 'LBDMONO', 'LBDNENO', 'LBDEONO',
    'LBDBANO', 'LBXRBCSI', 'LBXHGB', 'LBXHCT', 'LBXMCVSI',
    'LBXMCHSI', 'LBXMC', 'LBXRDW', 'LBXPLTSI', 'LBXMPSI'
]
rename_CBC = {
    'LBXWBCSI': 'White blood cell count (1000 cells/uL)',
    'LBXLYPCT': 'Lymphocyte percent (%)',
    'LBXMOPCT': 'Monocyte percent (%)',
    'LBXNEPCT': 'Segmented neutrophils percent (%)',
    'LBXEOPCT': 'Eosinophils percent (%)',
    'LBXBAPCT': 'Basophils percent (%)',
    'LBDLYMNO': 'Lymphocyte number (1000 cells/uL)',
    'LBDMONO': 'Monocyte number (1000 cells/uL)',
    'LBDNENO': 'Segmented neutrophils num (1000 cell/uL)',
    'LBDEONO': 'Eosinophils number (1000 cells/uL)',
    'LBDBANO': 'Basophils number (1000 cells/uL)',
    'LBXRBCSI': 'Red blood cell count (million cells/uL)',
    'LBXHGB': 'Hemoglobin (g/dL)',
    'LBXHCT': 'Hematocrit (%)',
    'LBXMCVSI': 'Mean cell volume (fL)',
    'LBXMCHSI': 'Mean cell hemoglobin (pg)',
    'LBXMC': 'Mean Cell Hgb Conc. (g/dL)',
    'LBXRDW': 'Red cell distribution width (%)',
    'LBXPLTSI': 'Platelet count (1000 cells/uL)',
    'LBXMPSI': 'Mean platelet volume (fL)'
}

df_CBC = df_CBC[columns_CBC]
df_CBC = df_CBC.rename(columns=rename_CBC)
df_CBC['SEQN'] = df_CBC['SEQN'].astype(int)
df_CBC = df_CBC.set_index('SEQN')
df_CBC.dropna(how='all', inplace=True)

In [13]:
df_medical = pd.merge(df_SBP, df_CBC, left_index=True, right_index=True, how='outer')
df_medical.to_csv('../processed_data/medical_table.csv')

### 4. User Tagging 

User taggings now contains two major parts: 

* Nutrition tags such `low_calorie` and `low_sodium`, which indicate nutrients the users needs.

* Status such as `hypertension` or `diabetic diet`, which indicate users' health conditions.

In [14]:
# Generate all tags and initialize them with 0. Following merging will update the tags. 
column_tag = [
    'low_calorie', 'high_calorie', 'low_carb', 'low_protein', 'high_protein' , 'low_saturated_fat', 'low_sugar', 'low_cholesterol', 'high_fiber',
    'low_sodium', 'high_potassium', 'high_iron', 'high_calcium', 'high_vitamin_d', 'high_vitamin_c', 'low_phosphorus', 'high_folate_acid', 'high_vitamin_b12'
]
# Create a new dataframe with the index (SEQN) from df_demo
df_tags = pd.DataFrame(index=df_demo.index)
# Initialize the nutritional tags in df_tag with 0
df_tags[column_tag] = 0


#### 4.1. Special Diet

In [15]:
# NHANES records if the user is on a special diet.
type_dietary = 'dietary'

df_TOT1 = concat_data_across_years(type_dietary, 'DR1TOT', years, year_char)
columns_diet = ["DRQSDT1", "DRQSDT2", "DRQSDT3", "DRQSDT4", "DRQSDT7", "DRQSDT8", "DRQSDT9", "DRQSDT10", "DRQSDT12"]
mappings_diet = {
    "DRQSDT1": "Weight loss/Low calorie diet", "DRQSDT2": "Low fat/Low cholesterol diet",
    "DRQSDT3": "Low salt/Low sodium diet", "DRQSDT4": "Sugar free/Low sugar diet", "DRQSDT7": "Diabetic diet",
    "DRQSDT8": "Weight gain/Muscle building diet", "DRQSDT9": "Low carbohydrate diet", "DRQSDT10": "High protein diet",
    "DRQSDT12": "Renal/Kidney diet"
}

df_diet = df_TOT1[["SEQN"] + columns_diet]
df_diet = df_diet.fillna(0)
df_diet.rename(columns=mappings_diet, inplace=True)
df_diet = df_diet.astype(int)
df_diet['SEQN'] = df_diet['SEQN'].astype(str)
df_diet = df_diet.set_index('SEQN')
# Convert all integers to 1s if a user is taking a diet
df_diet = (df_diet != 0).astype(int)
# Keep the records if there is at least one diet
df_diet = df_diet.loc[~(df_diet == 0).all(axis=1)]

columns_diet = df_diet.columns
# Convert the diets to health tags 
df_diet['low_calorie'] = df_diet['Weight loss/Low calorie diet']
df_diet['low_saturated_fat'] = df_diet['Low fat/Low cholesterol diet']
df_diet['low_sodium'] = df_diet['Low salt/Low sodium diet']
df_diet['low_sugar'] = df_diet['Sugar free/Low sugar diet']
df_diet['low_sugar'] = df_diet['Diabetic diet'] | df_diet['low_sugar']
df_diet['high_fiber'] = df_diet['Diabetic diet']
df_diet['low_carb'] = df_diet['Low carbohydrate diet']
df_diet['high_protein'] = df_diet['High protein diet']
df_diet['low_protein'] = df_diet['Renal/Kidney diet']
df_diet['low_sodium'] = df_diet['Renal/Kidney diet'] | df_diet['low_sodium']
df_diet['low_phosphorus'] = df_diet['Renal/Kidney diet']
df_diet['high_calorie'] = df_diet['Weight gain/Muscle building diet']
df_diet['high_protein'] = df_diet['Weight gain/Muscle building diet'] | df_diet['high_protein']

# Don't drop the columns, as they will serve as status column
# df_diet.drop(columns_diet, axis=1, inplace=True)

# Combine the health tags for each user
df_diet = df_diet.groupby('SEQN').max()

# The tags from the special diet are merged to df_tags, where full tag information is stored.
df_tags = merge_with_or(df_tags, df_diet)

#### 4.2. Medical Info

4.2.1.BMI & Waist Circumference

In [16]:
def tag_BMI_waist_circumference(row):
    underweight_BMI, overweight_BMI = 18.5, 30

    waist_threshold_male, waist_threshold_female = 102, 88
    waist_threshold = waist_threshold_male if row['gender'] == 1 else waist_threshold_female

    high_calories, low_calories = 0, 0
    if row['BMXBMI'] < underweight_BMI:
        high_calories = 1
    
    # In practice, either metrics can inidicate overweight. But for confidence, we use "and" here. 
    
    if row['BMXBMI'] >= overweight_BMI and row['BMXWAIST'] >= waist_threshold:
        low_calories = 1

    # this rarely happens, but it means the visceral fat is high, so the user still need low calories food.
    if high_calories == 1 and low_calories == 1:
        high_calories = 0

    # Obesity is marked 1 if the user is marked for low_calories
    obesity = 1 if low_calories == 1 else 0

    return high_calories, low_calories, obesity

type_table = 'examination'
df_BMI = concat_data_across_years(type_table, 'BMX', years, year_char)
df_BMI['SEQN'] = df_BMI['SEQN'].astype(int).astype(str)
df_BMI = df_BMI[['SEQN', 'BMXBMI', 'BMXWAIST']].copy()
# Gender information is needed to determine the waist threshold.
df_BMI = df_BMI.merge(df_demo, left_on='SEQN', right_index=True, how='left')
df_BMI[['high_calorie', 'low_calorie', 'obesity']] = df_BMI.apply(lambda row: tag_BMI_waist_circumference(row), axis=1, result_type='expand').astype(int)
df_BMI.set_index('SEQN', inplace=True)

# Take a look here for now. Later we will only pay attention the adults.
df_BMI = df_BMI.loc[df_BMI['age'] > 18]

df_tags = merge_with_or(df_tags, df_BMI[['high_calorie', 'low_calorie', 'obesity']])
df_main = df_demo.merge(df_BMI[['BMXBMI', 'BMXWAIST']], left_index=True, right_index=True, how='left')

4.2.2. Blood Pressure

In [20]:
def tag_blood_pressure(row):
    high_systolic_threshold, high_diastolic_threshold = 130, 80
    low_sodium, high_potassium, hypertension = 0, 0, 0

    # Check if the blood pressure is above the thresholds
    if row['Average_Systolic'] >= high_systolic_threshold or row['Average_Diastolic'] >= high_diastolic_threshold:
        low_sodium = 1
        high_potassium = 1
        hypertension = 1

    return low_sodium, high_potassium, hypertension

years = ['0304', '0506', '0708', '0910', '1112', '1314', '1516', '1718']
df_BP = concat_data_across_years(type_table, 'BPX', years, year_char)
df_BP['Average_Systolic'] = df_BP[['BPXSY1', 'BPXSY2', 'BPXSY3', 'BPXSY4']].mean(axis=1, skipna=True).fillna(-1).apply(np.floor).astype(int)
df_BP['Average_Diastolic'] = df_BP[['BPXDI1', 'BPXDI2', 'BPXDI3', 'BPXDI4']].mean(axis=1, skipna=True).fillna(-1).apply(np.floor).astype(int)

df_BP_O = concat_data_across_years(type_table, 'BPXO', ['1720'], year_char)
df_BP_O['Average_Systolic'] = df_BP_O[['BPXOSY1', 'BPXOSY2', 'BPXOSY3']].mean(axis=1, skipna=True).fillna(-1).apply(np.floor).astype(int)
df_BP_O['Average_Diastolic'] = df_BP_O[['BPXODI1', 'BPXODI2', 'BPXODI3']].mean(axis=1, skipna=True).fillna(-1).apply(np.floor).astype(int)

df_BP_concat = pd.concat([df_BP[['SEQN', 'Average_Systolic', 'Average_Diastolic']], df_BP_O[['SEQN', 'Average_Systolic', 'Average_Diastolic']]])
df_BP_concat['SEQN'] = df_BP_concat['SEQN'].astype(int).astype(str)

df_BP_concat[['low_sodium', 'high_potassium', 'hypertension']] = df_BP_concat.apply(tag_blood_pressure, axis=1, result_type='expand')
df_BP_concat.set_index('SEQN', inplace=True)

df_tags = merge_with_or(df_tags, df_BP_concat[['low_sodium', 'high_potassium', 'hypertension']])
df_main = df_main.merge(df_BP_concat[['Average_Systolic', 'Average_Diastolic']], left_index=True, right_index=True, how='left')

4.2.3. Low-Density Lipoprotein

In [22]:
type_table = 'laboratory'
year_char = 'D'
years = ['0506', '0708', '0910', '1112', '1314', '1516']
df_LDL = concat_data_across_years(type_table, 'TRIGLY', years, year_char)

years = ['1718', '1720']
year_char = 'J'
type_table = 'laboratory'
df_LDL_2 = concat_data_across_years(type_table, 'TRIGLY', years, year_char)

df_LDL_3 = concat_data_across_years(type_table, 'L13AM', ['0304'], 'C')

df_LDL = pd.concat([df_LDL[['SEQN', 'LBDLDLSI']], df_LDL_2[['SEQN', 'LBDLDLSI']], df_LDL_3[['SEQN', 'LBDLDLSI']]])
df_LDL['SEQN'] = df_LDL['SEQN'].astype(int).astype(str)

def tag_LDL(row):
    threshold = 3.3
    low_cholesterol, high_fiber, low_saturated_fat = 0, 0, 0
    if row['LBDLDLSI'] and row['LBDLDLSI'] > threshold:
        low_cholesterol, high_fiber, low_saturated_fat = 1, 1, 1

    return low_cholesterol, high_fiber, low_saturated_fat

df_LDL[['low_cholesterol', 'high_fiber', 'low_saturated_fat']] = df_LDL.apply(tag_LDL, axis=1, result_type='expand')
df_LDL.set_index('SEQN', inplace=True)

df_tags = merge_with_or(df_tags, df_LDL[['low_cholesterol', 'high_fiber', 'low_saturated_fat']])
df_main = df_main.merge(df_LDL['LBDLDLSI'], left_index=True, right_index=True, how='left')

4.2.4. Blood Urea Nitrogen

In [23]:
type_table = 'laboratory'
df_SBP = concat_data_across_years(type_table, 'BIOPRO',
                                 ['0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720'], 'D')
df_temp = concat_data_across_years(type_table, 'L40',
                                 ['0304'], 'C')
df_SBP = pd.concat([df_SBP, df_temp])[['SEQN', 'LBDSBUSI']]
df_SBP['SEQN'] = df_SBP['SEQN'].astype(int).astype(str)

def tag_protein(row):
    return 1 if row['LBDSBUSI'] >= 7.1 else 0

df_SBP['low_protein'] = df_SBP.apply(tag_protein, axis=1, result_type='expand')
df_SBP.set_index('SEQN', inplace=True)

# Do know that this is a weak indicator, as heightened BUN can be caused by many other factors, such as dehydration. 
df_tags = merge_with_or(df_tags, df_SBP[['low_protein']])
df_main = df_main.merge(df_SBP['LBDSBUSI'], left_index=True, right_index=True, how='left')

4.2.5. Opioid Misuse 

In [24]:
df_opioid = df_demo[df_demo['opioid_label'] == 1].copy()
df_opioid[['low_sugar', 'high_protein', 'high_fiber', 'opioid_misuse']] = 1
df_tags = merge_with_or(df_tags, df_opioid[['low_sugar', 'high_protein', 'high_fiber', 'opioid_misuse']])

4.2.6. Diabetes

In [25]:
type_table = 'laboratory'
df_glucose_1 = concat_data_across_years(type_table, 'BIOPRO',
                                ['0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720'], 'D')
df_glucose_2 = concat_data_across_years(type_table, 'L40',
                                ['0304'], 'C')
df_glucose = pd.concat([df_glucose_1, df_glucose_2])[['SEQN', 'LBDSGLSI']]

df_ghb_1 = concat_data_across_years(type_table, 'GHB',
                                  ['0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720'], 'D')
df_ghb_2 = concat_data_across_years(type_table, 'L10',
                                  ['0304'], 'C')
df_ghb = pd.concat([df_ghb_1, df_ghb_2])[['SEQN', 'LBXGH']]

df_diabete = df_glucose.merge(df_ghb, on='SEQN', how='outer')
df_diabete['SEQN'] = df_diabete['SEQN'].astype(int).astype(str)
df_diabete.set_index('SEQN', inplace=True)

def tag_diabetes(row):
    return 1 if (row['LBDSGLSI'] >= 7.0) & (row['LBXGH'] >= 6.5) else 0

df_diabete['low_sugar'] = df_diabete.apply(tag_diabetes, axis=1, result_type='expand')
df_diabete['high_fiber'] = df_diabete.apply(tag_diabetes, axis=1, result_type='expand')
df_diabete['diabetes'] = df_diabete.apply(tag_diabetes, axis=1, result_type='expand')
df_tags = merge_with_or(df_tags, df_diabete[['low_sugar', 'high_fiber', 'diabetes']])
df_main = df_main.merge(df_diabete[['LBDSGLSI', 'LBXGH']], left_index=True, right_index=True, how='left')

4.2.7. Red Blood Cell Count & Hemoglobin

In [26]:
type_table = 'laboratory'
df_blood_1 = concat_data_across_years(type_table, 'CBC',
                                ['0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720'], 'D')
df_blood_2 = concat_data_across_years(type_table, 'L25',
                                ['0304'], 'C')
df_blood = pd.concat([df_blood_1, df_blood_2])[['SEQN', 'LBXRBCSI', 'LBXHGB']]
df_blood['SEQN'] = df_blood['SEQN'].astype(int).astype(str)
df_blood.set_index('SEQN', inplace=True)
df_blood = df_blood.merge(df_demo, left_index=True, right_index=True, how='left')

def tag_RBC(row):
    low_threshold_male, low_threshold_female = 13.2, 11.6
    threshold = low_threshold_male if row['gender'] == 1 else low_threshold_female
    return 1 if (row['LBXRBCSI'] <= 4) & (row['LBXHGB'] < threshold) else 0

df_blood[['high_iron', 'high_vitamin_c', 'high_folate_acid', 'high_vitamin_b12']] = df_blood.apply(lambda row: [tag_RBC(row)] * 4, axis=1, result_type='expand')
df_tags = merge_with_or(df_tags, df_blood[['high_iron', 'high_vitamin_c','high_folate_acid', 'high_vitamin_b12']])
df_main = df_main.merge(df_blood[['LBXRBCSI', 'LBXHGB']], left_index=True, right_index=True, how='left')

4.2.8. Osteoporosis

In [27]:
type_table = 'questionnaire'
# Note that we don't actually have data for 11-12 and 15-16. 
df_ost = concat_data_across_years(type_table, 'OSQ', ['0304', '0506', '0708', '0910', '1112', '1314', '1516', '1718', '1720'], 'C')
df_ost = df_ost.drop_duplicates()
df_ost['SEQN'] = df_ost['SEQN'].astype(int).astype(str)
df_ost = df_ost.set_index('SEQN')
df_ost = df_ost[df_ost['OSQ060'] == 1]
df_ost[['high_calcium', 'high_vitamin_d', 'high_vitamin_c']] = 1
df_tags = merge_with_or(df_tags, df_ost[['high_calcium', 'high_vitamin_d', 'high_vitamin_c']])

In [28]:
df_tags = df_tags.astype(int)
df_tags.to_csv('../processed_data/user_tagging.csv')

### 5. Information Summary

In [29]:
# All related tables are merged to the main table.
df_main = df_main.merge(df_tags, left_index=True, right_index=True, how='left')

In [32]:
df_main.rename(columns={
    'BMXBMI': 'BMI',
    'BMXWAIST': 'Waist Circumference',
    'LBDLDLSI': 'LDL-cholesterol (mmol/L)',
    'LBDSBUSI': 'Blood urea nitrogen (mmol/L)',
    'LBDSGLSI': 'Glucose (mmol/L)',
    'LBXGH': 'Lycohemoglobin (%)',
    'LBXRBCSI': 'Red blood cell count (million cells/uL)',
    'LBXHGB': 'Hemoglobin (g/dL)'
}, inplace=True)

In [33]:
df_main.to_csv('../processed_data/user_info_data.csv')